# Importing packages

In [ ]:
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

# Defining data

In [2]:
# Reproducability
SEED = 4014
random.seed(SEED)
np.random.seed(SEED)

In [3]:
# Control variables
N_CASES = 19665  # total scam cases
TARGET_TOTAL_LOSS = 456_400_000  # approx $456.4M

In [6]:
# Report-derived distributions
# Age profile
AGE_DIST = {
    "<20": 0.055,
    "20-29": 0.192,
    "30-49": 0.356,
    "50-64": 0.247,
    "65+": 0.150,
}

# Average amount lost by age group
AGE_AVG_LOSS = {
    "<20": 4731,
    "20-29": 8541,
    "30-49": 22329,
    "50-64": 29434,
    "65+": 33672,
}

# Scam types by number of cases
SCAM_CASE_COUNTS = {
    "Phishing": 3779,
    "E-commerce": 3237,
    "Job": 2701,
    "Investment": 2698,
    "Government Impersonation": 1762,
    "Fake Friend Call": 1053,
    "Insurance Services": 791,
    "Sexual Services": 553,
    "Loan": 457,
    "Internet Love": 433,
}
SCAM_CASE_COUNTS["Others"] = N_CASES - sum(SCAM_CASE_COUNTS.values())

# Average amount lost by scam type
SCAM_AVG_LOSS = {
    "Phishing": 8057,
    "E-commerce": 2353,
    "Job": 21861,
    "Investment": 53915,
    "Government Impersonation": 71842,
    "Fake Friend Call": 2689,
    "Insurance Services": 27004,
    "Sexual Services": 2654,
    "Loan": 6660,
    "Internet Love": 27920,
    "Others": 13375,  # close to implied remainder
}

# Loss bucket shares
LOSS_BUCKET_DIST = {
    "<2k": 0.557,
    "2k-5k": 0.129,
    "5k-10k": 0.090,
    "10k-50k": 0.134,
    "50k-100k": 0.040,
    "100k+": 0.051,
}

LOSS_BUCKET_RANGES = {
    "<2k": (50, 2000),
    "2k-5k": (2000, 5000),
    "5k-10k": (5000, 10000),
    "10k-50k": (10000, 50000),
    "50k-100k": (50000, 100000),
    "100k+": (100000, 500000),
}

# Contact methods by total volume
CONTACT_COUNTS = {
    "Social Media": 5913,
    "Messaging": 4670,
    "Phone Call": 3204,
    "Online Shopping Platform": 1915,
    "Other Website": 992,
    "Email": 484,
    "Dating App/Website": 454,
    "SMS": 252,
    "Streaming Service": 172,
    "Classified Ads": 168,
}
CONTACT_DIST = {k: v / sum(CONTACT_COUNTS.values()) for k, v in CONTACT_COUNTS.items()}

# Platform splits where specified
PLATFORM_BY_CONTACT = {
    "Social Media": {
        "Facebook": 0.540,
        "TikTok": 0.238,
        "Instagram": 0.151,
        "Other Social": 0.071,
    },
    "Messaging": {
        "WhatsApp": 0.540,
        "Telegram": 0.363,
        "Facebook Messenger": 0.052,
        "Other Messaging": 0.045,
    },
    "Online Shopping Platform": {
        "Carousell": 0.711,
        "Facebook Marketplace": 0.209,
        "TikTok Shop": 0.027,
        "Other Shopping": 0.053,
    },
    "Phone Call": {
        "Phone": 1.0,
    },
    "Other Website": {
        "Website": 1.0,
    },
    "Email": {
        "Email": 1.0,
    },
    "Dating App/Website": {
        "Dating Platform": 1.0,
    },
    "SMS": {
        "SMS": 1.0,
    },
    "Streaming Service": {
        "Streaming Platform": 1.0,
    },
    "Classified Ads": {
        "Classifieds": 1.0,
    },
}

# Self-effected transfers
SELF_EFFECTED_BASE = 0.788

In [7]:
# Engineered conditional logic
# Age-conditioned scam type tendencies from report text.
AGE_SCAM_WEIGHTS = {
    "<20": {
        "E-commerce": 3.0,
        "Phishing": 1.5,
        "Job": 1.3,
    },
    "20-29": {
        "E-commerce": 2.8,
        "Job": 1.8,
        "Phishing": 1.2,
    },
    "30-49": {
        "E-commerce": 1.8,
        "Phishing": 1.8,
        "Job": 1.4,
        "Investment": 1.2,
    },
    "50-64": {
        "Phishing": 2.0,
        "Investment": 1.7,
        "Government Impersonation": 1.5,
    },
    "65+": {
        "Investment": 2.2,
        "Phishing": 1.8,
        "Government Impersonation": 1.7,
        "Insurance Services": 1.4,
    },
}

# Scam-specific contact tendencies derived from report descriptions
SCAM_CONTACT_WEIGHTS = {
    "Government Impersonation": {"Phone Call": 4.0, "Messaging": 2.0},
    "Investment": {"Messaging": 3.0, "Social Media": 2.0, "Other Website": 1.3},
    "Phishing": {"Social Media": 2.5, "Phone Call": 1.5, "Email": 1.2, "SMS": 1.2},
    "Insurance Services": {"Phone Call": 3.0, "Messaging": 2.0},
    "E-commerce": {"Online Shopping Platform": 3.5, "Messaging": 1.3, "Social Media": 1.3},
    "Internet Love": {"Dating App/Website": 4.0, "Messaging": 1.5},
    "Fake Friend Call": {"Phone Call": 2.5, "Messaging": 2.0},
    "Loan": {"Messaging": 1.8, "SMS": 1.5, "Social Media": 1.2},
    "Sexual Services": {"Messaging": 2.0, "Classified Ads": 1.8},
    "Job": {"Messaging": 2.5, "Social Media": 1.7},
    "Others": {},
}

# Scam-specific preferred platforms if contact type allows it
SCAM_PLATFORM_WEIGHTS = {
    "Government Impersonation": {"WhatsApp": 2.0, "Phone": 2.5},
    "Investment": {"Telegram": 2.5, "Facebook": 1.8},
    "Phishing": {"Facebook": 2.0, "TikTok": 1.8},
    "Insurance Services": {"WhatsApp": 2.0, "Phone": 2.2},
    "E-commerce": {"Carousell": 2.2, "Facebook Marketplace": 1.5},
}

# Scam-specific urgency levels
URGENCY_DIST = {
    "Government Impersonation": {"Low": 0.05, "Medium": 0.20, "High": 0.75},
    "Investment": {"Low": 0.10, "Medium": 0.45, "High": 0.45},
    "Phishing": {"Low": 0.20, "Medium": 0.55, "High": 0.25},
    "Insurance Services": {"Low": 0.10, "Medium": 0.35, "High": 0.55},
    "E-commerce": {"Low": 0.55, "Medium": 0.35, "High": 0.10},
    "Job": {"Low": 0.20, "Medium": 0.45, "High": 0.35},
    "Fake Friend Call": {"Low": 0.30, "Medium": 0.50, "High": 0.20},
    "Loan": {"Low": 0.20, "Medium": 0.50, "High": 0.30},
    "Internet Love": {"Low": 0.20, "Medium": 0.45, "High": 0.35},
    "Sexual Services": {"Low": 0.30, "Medium": 0.50, "High": 0.20},
    "Others": {"Low": 0.35, "Medium": 0.45, "High": 0.20},
}

# Scam-specific payment method tendencies
PAYMENT_DIST = {
    "Government Impersonation": {"Bank Transfer": 0.70, "Cash/Valuables": 0.20, "Card": 0.10},
    "Investment": {"Bank Transfer": 0.45, "Crypto": 0.40, "Card": 0.15},
    "Phishing": {"Card": 0.70, "Bank Transfer": 0.20, "E-wallet": 0.10},
    "Insurance Services": {"Bank Transfer": 0.70, "Card": 0.20, "Cash/Valuables": 0.10},
    "E-commerce": {"Bank Transfer": 0.45, "Card": 0.35, "E-wallet": 0.20},
    "Job": {"Bank Transfer": 0.60, "Crypto": 0.10, "Card": 0.10, "E-wallet": 0.20},
    "Fake Friend Call": {"Bank Transfer": 0.70, "PayNow/Transfer": 0.30},
    "Loan": {"Bank Transfer": 0.60, "E-wallet": 0.25, "Card": 0.15},
    "Internet Love": {"Bank Transfer": 0.70, "Crypto": 0.15, "Cash/Valuables": 0.15},
    "Sexual Services": {"Bank Transfer": 0.70, "E-wallet": 0.30},
    "Others": {"Bank Transfer": 0.60, "Card": 0.20, "E-wallet": 0.10, "Crypto": 0.10},
}

# Scam-specific tendency toward self-effected transfer
SELF_EFFECTED_BY_SCAM = {
    "Government Impersonation": 0.96,
    "Investment": 0.95,
    "Insurance Services": 0.93,
    "Job": 0.90,
    "Internet Love": 0.90,
    "Loan": 0.88,
    "E-commerce": 0.70,
    "Fake Friend Call": 0.85,
    "Sexual Services": 0.88,
    "Others": 0.80,
    "Phishing": 0.35,
}

# Utility functions

In [ ]:
# Utility function for weighted random choice
def weighted_choice(dist: Dict[str, float]) -> str:
    keys = list(dist.keys())
    probs = np.array(list(dist.values()), dtype=float)
    probs = probs / probs.sum()
    return np.random.choice(keys, p=probs)

In [ ]:
# Utility function to normalize a distribution
def normalize(d: Dict[str, float]) -> Dict[str, float]:
    s = sum(d.values())
    return {k: v / s for k, v in d.items()}

In [ ]:
# Convert case counts to probabilities for sampling
def make_case_count_distribution(case_counts: Dict[str, int]) -> Dict[str, float]:
    total = sum(case_counts.values())
    return {k: v / total for k, v in case_counts.items()}


SCAM_BASE_DIST = make_case_count_distribution(SCAM_CASE_COUNTS)

In [ ]:
# Sample age group based on distribution
def sample_age_group() -> str:
    return weighted_choice(AGE_DIST)

In [ ]:
# Sample scam type conditioned on age group
def sample_scam_type(age_group: str) -> str:
    probs = dict(SCAM_BASE_DIST)
    for scam, multiplier in AGE_SCAM_WEIGHTS.get(age_group, {}).items():
        probs[scam] *= multiplier
    probs = normalize(probs)
    return weighted_choice(probs)

In [ ]:
# Sample contact method conditioned on scam type
def sample_contact_method(scam_type: str) -> str:
    probs = dict(CONTACT_DIST)
    for method, multiplier in SCAM_CONTACT_WEIGHTS.get(scam_type, {}).items():
        probs[method] *= multiplier
    probs = normalize(probs)
    return weighted_choice(probs)

In [ ]:
# Sample platform conditioned on contact method and scam type
def sample_platform(contact_method: str, scam_type: str) -> str:
    platform_dist = dict(PLATFORM_BY_CONTACT[contact_method])
    for platform, multiplier in SCAM_PLATFORM_WEIGHTS.get(scam_type, {}).items():
        if platform in platform_dist:
            platform_dist[platform] *= multiplier
    platform_dist = normalize(platform_dist)
    return weighted_choice(platform_dist)

In [ ]:
# Sample urgency level conditioned on scam type
def sample_urgency(scam_type: str) -> str:
    return weighted_choice(URGENCY_DIST.get(scam_type, URGENCY_DIST["Others"]))

In [ ]:
# Sample payment method conditioned on scam type
def sample_payment_method(scam_type: str) -> str:
    return weighted_choice(PAYMENT_DIST.get(scam_type, PAYMENT_DIST["Others"]))

In [ ]:
# Sample whether user action was required for the transfer, conditioned on scam type
def sample_requires_user_action(scam_type: str) -> int:
    p = SELF_EFFECTED_BY_SCAM.get(scam_type, SELF_EFFECTED_BASE)
    return int(np.random.rand() < p)

In [ ]:
# Sample loss bucket conditioned on scam type and age group
def choose_loss_bucket(scam_type: str, age_group: str) -> str:

    probs = dict(LOSS_BUCKET_DIST)

    # Scam-type severity tilts
    if scam_type in {"Investment", "Government Impersonation"}:
        probs["10k-50k"] *= 1.6
        probs["50k-100k"] *= 2.1
        probs["100k+"] *= 3.3
        probs["<2k"] *= 0.45
        probs["2k-5k"] *= 0.65
    elif scam_type in {"Insurance Services", "Internet Love", "Job"}:
        probs["10k-50k"] *= 1.5
        probs["50k-100k"] *= 1.6
        probs["100k+"] *= 1.4
        probs["<2k"] *= 0.60
    elif scam_type in {"E-commerce", "Fake Friend Call", "Sexual Services"}:
        probs["<2k"] *= 1.7
        probs["2k-5k"] *= 1.3
        probs["50k-100k"] *= 0.35
        probs["100k+"] *= 0.20
    elif scam_type == "Phishing":
        probs["<2k"] *= 0.9
        probs["2k-5k"] *= 1.1
        probs["5k-10k"] *= 1.1
        probs["100k+"] *= 0.7

    # Age severity tilts
    if age_group in {"50-64", "65+"}:
        probs["10k-50k"] *= 1.2
        probs["50k-100k"] *= 1.4
        probs["100k+"] *= 1.5
        probs["<2k"] *= 0.75
    elif age_group in {"<20", "20-29"}:
        probs["<2k"] *= 1.2
        probs["2k-5k"] *= 1.1
        probs["50k-100k"] *= 0.8
        probs["100k+"] *= 0.65

    probs = normalize(probs)
    return weighted_choice(probs)

In [ ]:
# Sample amount lost based on loss bucket, with mild adjustments for scam type and age group to create more realistic variance within buckets
def sample_amount_from_bucket(bucket: str, scam_type: str, age_group: str) -> float:
    low, high = LOSS_BUCKET_RANGES[bucket]

    if bucket != "100k+":
        # Use a skewed beta so values concentrate lower within each bucket
        x = np.random.beta(a=1.2, b=3.0)
        amount = low + x * (high - low)
    else:
        # Heavy tail for 100k+ bucket
        amount = low + np.random.pareto(a=2.2) * 60_000
        amount = min(amount, high)

    # Mild scaling for high-severity scam types
    if scam_type == "Government Impersonation":
        amount *= np.random.uniform(1.00, 1.20)
    elif scam_type == "Investment":
        amount *= np.random.uniform(1.00, 1.15)
    elif scam_type == "E-commerce":
        amount *= np.random.uniform(0.85, 1.00)

    # Mild age scaling
    if age_group == "65+":
        amount *= np.random.uniform(1.05, 1.20)
    elif age_group == "<20":
        amount *= np.random.uniform(0.85, 1.00)

    return float(max(1, amount))

In [ ]:
# Calibration function to adjust amounts to better match target averages by scam type and age group, while preserving overall total loss
def calibrate_to_target_average(amounts: np.ndarray, scam_types: List[str], age_groups: List[str], alpha: float = 0.55) -> np.ndarray:
    
    df = pd.DataFrame({
        "amount": amounts,
        "scam_type": scam_types,
        "age_group": age_groups
    })

    # Global scale to target total loss
    global_scale = TARGET_TOTAL_LOSS / df["amount"].sum()
    df["amount"] *= global_scale

    # Scam-level adjustment toward target average
    for scam, target_avg in SCAM_AVG_LOSS.items():
        idx = df["scam_type"] == scam
        if idx.sum() > 0:
            current_avg = df.loc[idx, "amount"].mean()
            scam_scale = (target_avg / current_avg) ** alpha
            df.loc[idx, "amount"] *= scam_scale

    # Age-level adjustment toward target average
    for age, target_avg in AGE_AVG_LOSS.items():
        idx = df["age_group"] == age
        if idx.sum() > 0:
            current_avg = df.loc[idx, "amount"].mean()
            age_scale = (target_avg / current_avg) ** (alpha * 0.45)
            df.loc[idx, "amount"] *= age_scale

    # Final global rescale
    final_scale = TARGET_TOTAL_LOSS / df["amount"].sum()
    df["amount"] *= final_scale

    return df["amount"].to_numpy()

# Main Generator

In [ ]:
# Main function to generate synthetic scam data
def generate_synthetic_scam_data(n_cases: int = N_CASES) -> pd.DataFrame:
    rows = []

    for case_id in range(1, n_cases + 1):
        age_group = sample_age_group()
        scam_type = sample_scam_type(age_group)
        contact_method = sample_contact_method(scam_type)
        source_platform = sample_platform(contact_method, scam_type)
        urgency_level = sample_urgency(scam_type)
        payment_method = sample_payment_method(scam_type)
        requires_user_action = sample_requires_user_action(scam_type)
        loss_bucket = choose_loss_bucket(scam_type, age_group)
        transaction_amount = sample_amount_from_bucket(loss_bucket, scam_type, age_group)

        rows.append({
            "case_id": case_id,
            "age_group": age_group,
            "scam_type": scam_type,
            "contact_method": contact_method,
            "source_platform": source_platform,
            "requires_user_action": requires_user_action,
            "payment_method": payment_method,
            "urgency_level": urgency_level,
            "loss_bucket": loss_bucket,
            "transaction_amount": transaction_amount,
            "is_scam": 1,
        })

    df = pd.DataFrame(rows)

    # Calibrate the amounts
    df["transaction_amount"] = calibrate_to_target_average(
        df["transaction_amount"].to_numpy(),
        df["scam_type"].tolist(),
        df["age_group"].tolist(),
    )

    # Clean up and cap extremes
    df["transaction_amount"] = df["transaction_amount"].clip(lower=50, upper=750000).round(2)

    # Recompute bucket from calibrated amount
    df["loss_bucket"] = df["transaction_amount"].apply(assign_loss_bucket_from_amount)

    return df

# Helper function to assign loss bucket from amount, ensuring consistency after calibration
def assign_loss_bucket_from_amount(amount: float) -> str:
    if amount < 2000:
        return "<2k"
    elif amount < 5000:
        return "2k-5k"
    elif amount < 10000:
        return "5k-10k"
    elif amount < 50000:
        return "10k-50k"
    elif amount < 100000:
        return "50k-100k"
    return "100k+"

# Summary of data

In [ ]:
# Function to print summary statistics of the generated dataset
def summarize(df: pd.DataFrame) -> None:
    print("\n===== DATASET SUMMARY =====")
    print(f"Rows: {len(df):,}")
    print(f"Total loss: ${df['transaction_amount'].sum():,.2f}")
    print(f"Median loss: ${df['transaction_amount'].median():,.2f}")
    print(f"Self-effected share: {df['requires_user_action'].mean():.3f}")

    print("\nAge distribution:")
    print((df["age_group"].value_counts(normalize=True).sort_index() * 100).round(2))

    print("\nScam type distribution:")
    print((df["scam_type"].value_counts(normalize=True) * 100).round(2).head(12))

    print("\nLoss bucket distribution:")
    print((df["loss_bucket"].value_counts(normalize=True).sort_index() * 100).round(2))

    print("\nAverage loss by age group:")
    print(df.groupby("age_group")["transaction_amount"].mean().round(2).sort_index())

    print("\nAverage loss by scam type:")
    print(df.groupby("scam_type")["transaction_amount"].mean().round(2).sort_values(ascending=False))

# Create Synthetic Data

In [ ]:
df = generate_synthetic_scam_data(n_cases=N_CASES)
summarize(df)

output_path = "synthetic_spf_scam_cases_1H2025.csv"
df.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")